<a href="https://colab.research.google.com/github/Rimix-oups/milestones/blob/main/Copie_de_Copie_de_Copie_FINAL_de_26_11_2026_Project_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Group project — SeqTools: basic processing of biological sequences
**Python for Bioinformatics · MSc 1**

**Group name:** ...

| Member | Main responsibilities |
|---|---|
| Ameni Ben amar | ... |
| Rim Tlili  | ... |
| Dyhia Yefsah| ... |

**How to work together**
1. ONE member: `File → Save a copy in Drive`, rename it `GroupX_seqtools.ipynb`, then `Share` → add the other members
   as **Editors**.
2. Everyone works in this shared notebook, in the section of the current milestone. Colab saves automatically,
   but two people should not edit **the same cell** at the same time.
3. Keep the **contribution log** at the end up to date: it is used for the individual part of the grade.

The full instructions are in the project brief (`Project_SeqTools.pdf`).

## ⚙️ Setup — run these cells every time you open the notebook
The Colab disk is erased when the session ends: these cells re-create the data file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile sequences.fasta
>gene_alpha synthetic gene, one ORF on the forward strand
AGGTAAGGTGGTTGAGATCTGGACTTTTGACGCCTGGAGCATGCCGCAGTGCTCCTCGAA
AAGTAGCCATGCCTTGGGCTGCTCCCAGGCGCTCCGTTGTTCTAGTGCTTTCACCAGCCA
AGTATTGGACGTTCGAAGTGTTTCCAATACCCCCATGCCGGACATAGTCTGTATCCCACC
CCTCCCCATTTTATTGCTTGTCAGGGAGGGACCGACTAGTGAGGCTCCCACTTCAAAAGG
TCTGGGTCACTTGTGTCTCGCAGCTTATCTGCGGAGAACGGAGTAGCCTGTAAGGGTTAA
GCTAGAAATTTCGGATGAAAGAGTGCTGCTCCCTTG
>gene_beta synthetic gene, ORF on the reverse strand
CACCACTGCAACGCGATGTGCACGTTGTAATTAACACAACCCTGCGCAACAACCCCAACG
CCGGAGGTCGGGGGGCGAGCCTATTCCGTCCGAATACTCGCCCCGACCCTGAAACCTCGT
GTCGCCACCGCTGGCGCCGCTGGCGAGTGGCCGGCAGCAAGTGGCAGCGTCGCAATGAAG
GCGGTGGTATGTGCCGTACCCTTTGTCCACCGACACAGTGGATCCGCGCATAACTAATTG
AGTCGGGAGGAATTGTTCCTATGAAG
>gene_gamma synthetic gene, lower case, two ORFs
tgcgttcgtagctagcactgtcagtatgagaccgtgccacgtgctaatcggaacttcgat
gcgccccctaaaccgaggttacgttttcaagcatttggacacattggtcatagcaatcaa
aaaccgcaggtctcttgctaggtggctttctaataccctagggtaatgattgtgttttgt
aagcagatgatagggggcacgcaacagaaggccgaacgattttcggccaactccggcggg
acacgggcgtcgggactgggaagcccccagagatcacctgtcgcccagaaggtctgagaa
cggcgcgtgctagagaacgcccacagc
>gene_delta contains ambiguous bases (N)
CCTGGTTCGTGTCCANNNTTTGCTTAAAGCTCAAAAAAGGATGTGCCTAACAGAATTGCG
NCACTATTCTAGCTCACGTCCAGGCCATGTCTTGCTCGGGCTTCGGACAGGCGTATGCCG
TCTATGCCTGGCCCCACTCGGGCTCCCCCCGTCTTACCCTGAATAAAACTGCGGGAATAG
AATTTG
>gene_epsilon nested start codons
TTCTTTGGCAGAATACGCAGATGACTGTTCCGCCTGGGATCCGAATAAATATGATCTACC
TTAGAGACCGTAAGTTAACACGCAACAGATTTAGTACGACCCTTACCGCCCATAGACATA
CGCCGACGGTCCGCCGCGCCTACGGAAGCGGCAACGCATCATTAAGATCTGACTTGGGGC
TTATGTGTGACATTGCCTTGTGGTAGCGTGCCCGGGTATCATGCGAGATTA
>gene_zeta short fragment, no ORF
CCGTTAGCATTAGC
>gene_eta typing error in the sequence
CAAAATCATGCTACACTCGGACGTTCAAAAGTACGAAAACACATAGTATCATGCCX1AGT
TTCACCGGGATATACTTGACCTAGTGTTTGCCCTACT
>gene_theta empty record
>gene_iota GC-rich synthetic gene
GACGGTGGCGACCTGGCCGGATGCGGCAGGTAGGAGCCGACCCTGGGGAATATGGAAATA
ACGCCGTCTTCTGTCAGATCAAGCAACCCGGGCCACCCCATGAGTACTCAGTCTCGACCG
GCGGCCCATTACCCCCGAGAGGGTCGTACGGACGGTACCGGGCCTGCCCACCGGCGCGCG
AGGTGCCGTAGCCTGAGCACAGCCGACGCGT

Writing sequences.fasta


In [ ]:
import os
os.makedirs("data", exist_ok=True)
os.replace("sequences.fasta", "data/sequences.fasta")
print(os.listdir("data"))

['sequences.fasta']


In [ ]:
# Standard genetic code (NCBI table 1). '*' = stop codon
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

TEST_SEQ = "ATGTGCTCATGCCTCTTCACATACTTCCGCAGGCGATACTAA"   # small test sequence for the first milestones
print(len(GENETIC_CODE), "codons")

64 codons


---
## Milestone 1 — Get organised and first computations (session 1)

1. Form **groups of 3–4**. One person opens `00_Project_Starter.ipynb`, saves a copy in Drive
   and **shares it** with the group (Share → Editor). Rename it `GroupX_seqtools.ipynb`.
2. Read the project brief together (`Project_SeqTools.pdf`), and split the work.
3. In the starter notebook, section *Milestone 1*: store the test sequence in a variable and use
   **variables and operators only** to compute:
   - the length of the sequence
   - the number of A, C, G and T (you may use `seq.count("A")`)
   - the GC content in %
   - whether it starts with `"ATG"` (`in`, or slicing, which we will see in session 4)

In [ ]:
sequence = "ATTTGCTCATGCCTCTTCACATACTTCCGCAGGCGATACTAA"

# ========length of the sequence ==============Ameni========

length = len(sequence)
print("the length of the sequence is ", length)

#========number of each base =================Dihya==========

A = sequence.count("A")
C = sequence.count("C")
G = sequence.count("G")
T = sequence.count("T")
print("number of A is", A)
print("number of C is", C)
print("number of T is", T)
print("number of G is", G)

#===========GC content===================Rim================



#=========== start codon===============Ameni===============
# use slicing

first_codon = sequence[0:3] # python here sees the character from position 0 to position 2( 3 cacarcters !) . 3 not included
if first_codon == "ATG":
  print ("the sequence starts with ATG ")
else :
  print("the sequence does not start with ATG")



the length of the sequence is  42
number of A is 10
number of C is 13
number of T is 13
number of G is 6
the GC content is  45.23809523809524 %
the sequence does not start with ATG


---
## Milestone 2 — Validation with loops and conditions (session 2)

In the group notebook, section *Milestone 2*:
1. **Task 1 (logic only):** loop over the characters of a sequence and decide if it is a valid DNA
   sequence (only A, C, G, T, upper or lower case). Print `True`/`False`. Test it on `"atggagataac"` and
   `"ATGCXAGA"`.
2. Classify the GC content as low / medium / high with `if/elif/else`.
3. Use `range(0, len(seq), 3)` to print the sequence codon by codon.

In [ ]:
sequence="atggCGCGCGCtCCGCCCc"

#===========validate DNA===============Ameni=============

seq=sequence.upper()
valid_dna = True
for i in range (len(seq)):
  if seq[i] not in ("A","G","C","T"):
    valid_dna=False
  break
print("VALID DNA SEQUENCE")

#============Classify the GC content==========Rim=========

# first, we need to calulate the gc content if the sequence is a valid DNA sequence
if valid_dna:
  gc_content = seq.count("G")+ seq.count("C")  #this counts the total of g and c in the whole sequence
  gc_perc=(gc_content/len(seq))*100            #this calculates the percentage of g and c in the whole sequence
  print ("GC CONTENT IS", gc_perc,"%")
  # now let's classify the GC content according to the value we computed
  if gc_perc<30:
    print("GC content is low")
  elif gc_perc<70:
    print ("GC content is medium")
  else :
    print("GC content is high")

#=============print the sequence codon by codon===========Dihya============

  for i in range (0,len(seq)-2,3): #0 is the start position ,len(seq) is the final position and 3 is the step
    print(seq[i:i+3])







VALID DNA SEQUENCE
GC CONTENT IS 84.21052631578947 %
GC content is high
ATG
GCG
CGC
GCT
CCG
CCC


---
## Milestone 3 — Dictionaries for frequencies and the genetic code (session 3)

In the group notebook, section *Milestone 3*:
1. **Task 2:** build a dictionary with the frequency of each symbol in a sequence.
2. Load the `GENETIC_CODE` dictionary (in the starter) and translate the test sequence codon by codon.
3. **Task 3 (draft):** count the amino acids of a protein sequence and print them from most to least frequent
   (`sorted(..., key=..., reverse=True)`).
4. **Task 9 (draft):** for one amino acid (e.g. `'L'`), count how often each of its codons is used.

In [ ]:
sequence='TTTTGGtc'
#========a dictionary with the frequency of each base in a sequence============ Ameni

frequency={} # this creates an empty dictionary, we will progressively fill it with nucleotides and their counts.
for i in sequence: # looping through the sequence
  if i in frequency: # Checks whether the nucleotide already exists in the dictionary
      frequency[i]+=1 # if it does exist , increase its count with 1
  else :
      frequency[i]=1 # Otherwise, creates a new key and give it a count 1
print(" the frequency of each nucleotide", frequency)



#===========loading the genetic code and translating sequence codon by codon========Dihya
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}
protein_sequence='' # this creates a an empty string that we're going to fill
for i in range (0,len(sequence)-2,3): #goes through the whole sequence with a step of 3
  codon = sequence[i:i+3] # assigns each time the 3 bases to the codon variable
  protein_sequence += GENETIC_CODE [codon] #translate the codon into a protein given in the genetic code that we loaded
  # and add it to the protein sequence so that the sequence will be shown in one line.
print(" the protein sequence is ", protein_sequence)



#============count the AA of a protein sequence and print them from most to least frequent=======Rim
AA_freq = {}
for i in protein_sequence:
  if i in AA_freq:
      AA_freq[i] += 1
  else: AA_freq[i] =1
sorted_AAseq = sorted(AA_freq.items(), key=lambda x: x[1], reverse=True)
#Sort the dictionary AA_seq items (amino acid, frequency)
#lambda x: x[1] means to use the second element of each pair (the frequency) for sorting
#reverse=True sorts the frequencies in descending order (highest to lowest)
print(sorted_AAseq)



# ==========count how often a codon for a specific AA is used========Ameni


# first let's create an empty dictionary to store the codons coding for leucine and their frequencies.
leucine = {"TTA": 0, "TTG": 0, "CTT": 0,"CTC": 0, "CTA": 0, "CTG": 0}
#then go through all codons in the sequence
for i in range(0,len(sequence)-2,3):
  codon= sequence[i:i+3]
  #if the codon is present in leucine dictionary, then increase the frequency of this specific leucine codon by 1
  if codon in leucine:
    leucine[codon]+=1
print("frequency of codons for leucine ",leucine)













 the frequency of each nucleotide {'T': 4, 'G': 2, 't': 1, 'c': 1}
 the protein sequence is  FW
[('F', 1), ('W', 1)]
frequency of codons for leucine  {'TTA': 0, 'TTG': 0, 'CTT': 0, 'CTC': 0, 'CTA': 0, 'CTG': 0}


---
## Milestone 4 — String processing: transcription, reverse complement, frames (session 4)

In the group notebook, section *Milestone 4*:
1. **Task 4:** GC content with string methods.
2. **Task 5:** transcription (`replace`).
3. **Task 6:** reverse complement.
4. **Tasks 7–8:** translate from a given start position.
5. **Task 10 (draft):** build the 6 reading frames (3 forward and 3 on the reverse complement).

In [ ]:
# =================GC content with string methods============Ameni

sequence="TTTCCCGGG"
seq=sequence.upper()
print("the sequence",seq)
count_GC= seq.count("G")+seq.count("C")
print(count_GC)

#=============transcription using replace==========Rim
RNA= seq.replace("T","U")
print("RNA",RNA)

#=============reverse complement============Dihya


#first , let's Create a translation table to replace each nucleotide with its complementary nucleotide
table = str.maketrans("ACGT", "TGCA")
#Replace each nucleotide in the sequence with its complement
comp = seq.translate(table)
print("complement sequence",comp)
# reverse the complement sequence
rc=comp[::-1]
print("reverse complement ",rc)

#or we can reverse the sequence first and then make the complementary for that sequence


#==============translate from a given start position.first let's define a starting position==========Ameni
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

start = 2
#then define the coding area of the RNA that we're gonna translate starting from the position that we defined
coding_seq=seq[start:]
print("coding DNA",coding_seq)
# now let's translate into proteins using the genetic code that we have
protein_sequence=''
for i in range(0,len(coding_seq)-2,3):
  codon=coding_seq[i:i+3]
  protein_sequence += GENETIC_CODE[codon]
print("the protein sequence starting from position ", start, " is  : ", protein_sequence)

#=======build the 6 reading frames (3 forward and 3 on the reverse complement)======Rim
for i in range (3):
  forward=""
  rv=""
  for j in range(i,len(seq)-2,3):
    forward+=seq[j:j+3]
    rv+=rc[j:j+3]
  print("reading frame  + ", i, forward )
  print("reading frame  - ", i, rv)


















the sequence TTTCCCGGG
6
RNA UUUCCCGGG
complement sequence AAAGGGCCC
reverse complement  CCCGGGAAA
coding DNA TCCCGGG
the protein sequence starting from position  2  is  :  SR
reading frame  +  0 TTTCCCGGG
reading frame  -  0 CCCGGGAAA
reading frame  +  1 TTCCCG
reading frame  -  1 CCGGGA
reading frame  +  2 TCCCGG
reading frame  -  2 CGGGAA


---
## Milestone 5 — Refactor everything into functions (session 5)

In the group notebook, section *Milestone 5*:
1. Turn every piece of code you wrote into a **function with a docstring** (purpose, inputs, output).
   Use the names from the brief (`validate_dna`, `frequency`, `gc_content`, `transcription`, ...).
2. `transcription` must **raise a `ValueError`** if the DNA is not valid.
3. **Tasks 11–14:** `all_proteins`, `has_stop_codon`, `validate_protein`, `all_orfs`.
4. Write at least one `assert` test per function (e.g. `assert validate_dna("ATGC") == True`).

In [ ]:

def validate_dna(seq):
  """ this function checks wether the input sequence is a DNA  sequence or not.
  The function returns True or False  """
  seq = seq.upper()
  for i in range(len(seq)):
    if seq[i] not in "ACTG" :
      return False
  return True
help(validate_dna)
# we use assert with known result to make sure the function is reliable before using it to discover unknown results.
assert validate_dna("ATGC") == True


def frequency(seq):
    """Computes the frequency of the different symbols in a sequence.
    The result is a dictionary whose keys are the symbols
    and whose values are their frequencies.
    """
    seq = seq.upper()
    frequencies = {
        "A": seq.count("A"),
        "C": seq.count("C"),
        "T": seq.count("T"),
        "G": seq.count("G")
    }

    return frequencies

help(frequency)
# Verify that the function produces the expected result
assert frequency("ATGCGCTGAC") == {'A': 2, 'C': 3, 'T': 2, 'G': 3}



def aa_frequency(protein):
  """  computes the frequency of the amino acids in a protein
sequence. The result is a dictionary whose keys are the amino acids and whose values are their frequencies. """

  protein = protein.upper()
  aa_freq={}
  for AA in protein:
    if AA in aa_freq:
      aa_freq[AA]+=1
    else : aa_freq[AA]=1
  return aa_freq
help(aa_frequency)
assert aa_frequency("FMMATTIFHH")== {'F': 2, 'M': 2, 'T': 2, 'I': 1, 'H': 2,'A':1}


def gc_content(seq):
  """ this function computes the GC content (%) of a DNA sequence. the result is in percent and rounded to 2 decimals """
  seq=seq.upper()
  return round((seq.count("G")+seq.count("C"))/len(seq)*100,2)
help(gc_content)
assert gc_content("ATGC")==50.0


def transcription(seq):
  """ this function translates the dna sequence into an rna sequence by replacing T with U """
  seq = seq.upper()
  if validate_dna(seq):# before replacing, we need to check wether the input sequence is a dna sequence or not
    rna = seq.replace("T", "U")
    return rna
  else :
    raise ValueError("Invalid DNA sequence")
    #this is a way of telling Python:The input is invalid, so stop executing this function and report the error mentionned
help(transcription)
assert transcription("AATTGGCC")== "AAUUGGCC"


def reverse_complement(seq):
  """ this function retruns the reverse complement of an input sequence: it replaces each
  nucleotide with its complementary base then reads the complementary sequence from right to left """
  seq=seq.upper()
  table =str.maketrans("ATCG","TAGC")
  comp_seq =seq.translate(table)
  rev_comp_seq =comp_seq[::-1]
  return rev_comp_seq
help(reverse_complement)
assert reverse_complement("ATGC")== "GCAT"



def translate_codon(codon):
    """Return the amino acid encoded by an RNA codon.
    Return STOP for a stop codon and X for an unknown codon.
    """
    GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}
    codon = codon.upper()
    if codon in GENETIC_CODE:
        if GENETIC_CODE[codon] == '*':
            return "STOP"
        else:
            return GENETIC_CODE[codon]
    else:
        return "X"
help(translate_codon)
assert translate_codon("agc")== "S"


def translate(seq, start=0):
  """ this function turns a dna sequence into an amino acid sequence.
  0 is the default value, but you can change it whenever you call the function."""
  protein_seq=""
  seq=seq.upper()
  for i in range(start,len(seq)-2,3):
    codon=seq[i:i+3]
    protein_seq+=translate_codon(codon)
  return protein_seq
help(translate)
assert translate("ATTTGCTCATG",2)== "LLM"

def validate_protein(protein):
  """ this function tells you wether the input protein sequence is a protein sequence or not """
  GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}
  protein=protein.upper()
  for i in protein:
      if i not in GENETIC_CODE.values():
         return "NON VALID PROTEIN SEQUENCE"
  return "VALID PROTEIN SEQUENCE"
help(validate_protein)
assert validate_protein("FMMATTxFHH")== "NON VALID PROTEIN SEQUENCE"


def has_stop_codon(seq,start=0):
  """ this function checks wether the input DNA sequence has a STOP codon or not. and you can choose the position of start"""
  seq=seq.upper()
  for i in range (start, len(seq)-2,3):
      if seq[i:i+3] in ("TAG", "TAA", "TGA"):
        return "the sequence has a STOP codon"
  return "the sequence does not have a STOP codon"
help(has_stop_codon)
assert has_stop_codon("ATAATTTGa",0)== "the sequence has a STOP codon"

def find_ORFs(seq):
  """ an ORF is open reading frame wich is a DNA sequence that starts with a start condon and ends with STOP codon
  this function finds all the ORFs in a DNA sequence and returns the potential ORF sequences  """
  seq=seq.upper()
  start_codon = "ATG"
  stop_codons = ["TAG", "TAA", "TGA"]
  if validate_dna(seq):
    ORFs = []
    #we need 2 loops: one searches for the start codon and searches for the stop codon after a start codon
    for i in range(len(seq)-2): # we want to find ATG at every possible position.
      if seq[i:i+3]==start_codon: # if he finds a start codon , then we need to loop over from that position until we find a stop codon

        for j in range(i+3,len(seq)-2,3):
          if seq[j:j+3] in stop_codons: # if we find a stop codon then the list gets a string from i position to the j+3 position
            ORFs.append(seq[i:j+3])

            break

    return ORFs

  else :
    raise ValueError("Invalid DNA sequence")#if the input is not valid dna sequence , the code won't work!
help(find_ORFs)
assert find_ORFs("ATGTAACCC")== ["ATGTAA"]

def codon_usage(dna, aa):
    """Return a dictionary containing the percentage of use
    of each DNA codon encoding a specific amino acid,
    rounded to 2 decimal places.
    """
    dna = dna.upper()
    aa = aa.upper()
    counts = {}
    total = 0 # this is useful to compute the percentage of use after

    for i in range(0, len(dna)-2, 3): # loop over each codon
        codon = dna[i:i+3]

        if translate_codon(codon) == aa: # if the result of the function translate_codon == AA put as input
            total += 1 # the usage of this AA gets higher with 1

            if codon in counts: # if the dictionnnary has the AA then add 1 to it's value
                counts[codon] += 1
            else: # if not , just put 1 as value
                counts[codon] = 1
# so now we've created a dictionnary that has different codons for the same AA and their frequencies
# now we will wreate another dictionnary that has the percentage of use not frequency
    usage = {}
    for codon in counts: # loop over each codon in the count dictionnary
        usage[codon] = round(counts[codon] / total * 100, 2) # the value of codon in usage dictionary is the value of the same codon in count dictionary / total
    return usage
help(codon_usage)
assert codon_usage("CTGCTGTTAGCC", "L") == {"CTG": 66.67, "TTA": 33.33}























Help on function validate_dna in module __main__:

validate_dna(seq)
    this function checks wether the input sequence is a DNA  sequence or not.
    The function returns True or False

Help on function frequency in module __main__:

frequency(seq)
    Computes the frequency of the different symbols in a sequence.
    The result is a dictionary whose keys are the symbols
    and whose values are their frequencies.

Help on function aa_frequency in module __main__:

aa_frequency(protein)
    computes the frequency of the amino acids in a protein
    sequence. The result is a dictionary whose keys are the amino acids and whose values are their frequencies.

Help on function gc_content in module __main__:

gc_content(seq)
    this function computes the GC content (%) of a DNA sequence. the result is in percent and rounded to 2 decimals

Help on function transcription in module __main__:

transcription(seq)
    this function translates the dna sequence into an rna sequence by replacing T 

In [ ]:
#option 2 pour fonction 2
seq= TEST_SEQ
freq={}
for i in seq:
  if i in freq:
    freq[i]+=1
  else: freq[i]=1
print(freq)

# option 2 pour gc content
def gc_content(seq):
  return (seq.count("G")+seq.count("C"))/len(seq)*100
print(gc_content(TEST_SEQ))

#option 2 pour fonction 5

reverse_seq = seq[::-1]

resultat = ""

for nt in reverse_seq:
    if nt == "A":
        resultat += "T"
    elif nt == "T":
        resultat += "A"
    elif nt == "C":
        resultat += "G"
    elif nt == "G":
        resultat += "C"

print("Reverse complement:", resultat)
print("                   ", TEST_SEQ)

#option 2 pour genetic code codon
list_aa=[]
for i in range(0, len(seq) - 2, 3):  # -2 given by AI: # Le -2 évite de commencer un codon s'il reste moins de 3 nucléotides à lire.
    codon = seq[i:i+3]
    aa = GENETIC_CODE[codon]
    list_aa.append(aa)

seq_aa = "".join(list_aa)
print(seq_aa)


#option 2 pour sequence aa
def translate(seq, start=0):
  aa_seq=""
  for i in range(0, len(seq) - 2, 3):
    aa_seq+=translate_codon(seq[i:i+3])
  return aa_seq
print(translate(TEST_SEQ))


#option 2 pour fonction orf
def all_orfs(seq):
    """Recherche les ORF dans une séquence ADN."""
    orfs = []
    for start in range(3):
        for i in range(start, len(seq) - 2, 3):
            codon = seq[i:i+3]
if codon == "ATG":
    debut = i





{'A': 10, 'T': 12, 'G': 7, 'C': 13}
47.61904761904761
Reverse complement: TTAGTATCGCCTGCGGAAGTATGTGAAGAGGCATGAGCACAT
                    ATGTGCTCATGCCTCTTCACATACTTCCGCAGGCGATACTAA
MCSCLFTYFRRRY*
MCSCLFTYFRRRYSTOP


In [ ]:
def frequency(seq):
    """ computes how many of each base in a sequence """
    frequencies = {
        "A": seq.count("A"),
        "C": seq.count("C"),
        "T": seq.count("T"),
        "G": seq.count("G")
    }
    return frequencies

print(frequency("AUGCA"))
help(frequency)



{'A': 2, 'C': 1, 'T': 0, 'G': 1}
Help on function frequency in module __main__:

frequency(seq)
    computes how many of each base in a sequence



ameni : validate dna / gc count/translate codon /find_orf/ write summary
rim : frequency / transcription /translate/ stop_codon/ write
dihya : aa fraquency/reverse complement/validate protein / codon usage / read fasta

---
## Milestone 6 — Module + main program (session 6)

In the group notebook, section *Milestone 6*:
1. Move all your functions into **`seqtools.py`** with `%%writefile seqtools.py`.
2. Write **`main.py`**: it does `import seqtools`, asks the user for a DNA sequence with `input()`,
   checks it and prints a small report (use `datetime` to add the date and time of the analysis).
3. Run it from the notebook: `!python main.py`.
4. Tip: put your module under a `if __name__ == "__main__":` guard if it has test code.

In [ ]:
%%writefile seqtools.py
def validate_dna(seq):
  """ this function checks wether the input sequence is a DNA  sequence or not.
  The function returns True or False  """
  seq = seq.upper()
  for i in range(len(seq)):
    if seq[i] not in "ACTG" :
      return False
  return True
if __name__ == "__main__":
  help(validate_dna)
# we use assert with known result to make sure the function is reliable before using it to discover unknown results.
  assert validate_dna("ATGC") == True



def frequency(seq):
    """Computes the frequency of the different symbols in a sequence.
    The result is a dictionary whose keys are the symbols
    and whose values are their frequencies.
    """
    seq = seq.upper()
    frequencies = {
        "A": seq.count("A"),
        "C": seq.count("C"),
        "T": seq.count("T"),
        "G": seq.count("G")
    }

    return frequencies
if __name__ == "__main__":
  help(frequency)
# Verify that the function produces the expected result
  assert frequency("ATGCGCTGAC") == {'A': 2, 'C': 3, 'T': 2, 'G': 3}



def aa_frequency(protein):
  """  computes the frequency of the amino acids in a protein
sequence. The result is a dictionary whose keys are the amino acids and whose values are their frequencies. """

  protein = protein.upper()
  aa_freq={}
  for AA in protein:
    if AA in aa_freq:
      aa_freq[AA]+=1
    else : aa_freq[AA]=1
  return aa_freq
if __name__ == "__main__":
  help(aa_frequency)
  assert aa_frequency("FMMATTIFHH")== {'F': 2, 'M': 2, 'T': 2, 'I': 1, 'H': 2,'A':1}


def gc_content(seq):
  """ this function computes the GC content (%) of a DNA sequence. the result is in percent and rounded to 2 decimals """
  seq=seq.upper()
  if len(seq) == 0:
    return ("ADN sequence empty.")
  else:

    return round((seq.count("G")+seq.count("C"))/len(seq)*100,2)
if __name__ == "__main__":
  help(gc_content)
  assert gc_content("ATGC")==50.0


def transcription(seq):
  """ this function translates the dna sequence into an rna sequence by replacing T with U """
  seq = seq.upper()
  if validate_dna(seq):# before replacing, we need to check wether the input sequence is a dna sequence or not
    rna = seq.replace("T", "U")
    return rna
  else :
    raise ValueError("Invalid DNA sequence")
    #this is a way of telling Python:The input is invalid, so stop executing this function and report the error mentionned
if __name__ == "__main__":
  help(transcription)
  assert transcription("AATTGGCC")== "AAUUGGCC"


def reverse_complement(seq):
  """ this function retruns the reverse complement of an input sequence: it replaces each
  nucleotide with its complementary base then reads the complementary sequence from right to left """
  seq=seq.upper()
  table =str.maketrans("ATCG","TAGC")
  comp_seq =seq.translate(table)
  rev_comp_seq =comp_seq[::-1]
  return rev_comp_seq
if __name__ == "__main__":
  help(reverse_complement)
  assert reverse_complement("ATGC")== "GCAT"



def translate_codon(codon):
    """Return the amino acid encoded by an RNA codon.
    Return STOP for a stop codon and X for an unknown codon.
    """
    GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}
    codon = codon.upper()
    if codon in GENETIC_CODE:
        if GENETIC_CODE[codon] == '*':
            return "STOP"
        else:
            return GENETIC_CODE[codon]
    else:
        return "X"
if __name__ == "__main__":
  help(translate_codon)
  assert translate_codon("agc")== "S"


def translate(seq, start=0):
  """ this function turns a dna sequence into an amino acid sequence.
  0 is the default value, but you can change it whenever you call the function."""
  protein_seq=""
  seq=seq.upper()
  for i in range(start,len(seq)-2,3):
    codon=seq[i:i+3]
    protein_seq+=translate_codon(codon)
  return protein_seq
help(translate)
assert translate("ATTTGCTCATG",2)== "LLM"

def validate_protein(protein):
  """ this function tells you wether the input protein sequence is a protein sequence or not """
  GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}
  protein=protein.upper()
  for i in protein:
      if i not in GENETIC_CODE.values():
         return "NON VALID PROTEIN SEQUENCE"
  return "VALID PROTEIN SEQUENCE"
help(validate_protein)
assert validate_protein("FMMATTxFHH")== "NON VALID PROTEIN SEQUENCE"


def has_stop_codon(seq,start=0):
  """ this function checks wether the input DNA sequence has a STOP codon or not. and you can choose the position of start"""
  seq=seq.upper()
  for i in range (start, len(seq)-2,3):
      if seq[i:i+3] in ("TAG", "TAA", "TGA"):
        return "the sequence has a STOP codon"
  return "the sequence does not have a STOP codon"
help(has_stop_codon)
assert has_stop_codon("ATAATTTGa",0)== "the sequence has a STOP codon"

def find_ORFs(seq):
  """ an ORF is open reading frame wich is a DNA sequence that starts with a start condon and ends with STOP codon
  this function finds all the ORFs in a DNA sequence and returns the potential ORF sequences  """
  seq=seq.upper()
  start_codon = "ATG"
  stop_codons = ["TAG", "TAA", "TGA"]
  if validate_dna(seq):
    ORFs = []
    #we need 2 loops: one searches for the start codon and searches for the stop codon after a start codon
    for i in range(len(seq)-2): # we want to find ATG at every possible position.
      if seq[i:i+3]==start_codon: # if he finds a start codon , then we need to loop over from that position until we find a stop codon

        for j in range(i+3,len(seq)-2,3):
          if seq[j:j+3] in stop_codons: # if we find a stop codon then the list gets a string from i position to the j+3 position
            ORFs.append(seq[i:j+3])

            break

    return ORFs

  else :
    raise ValueError("Invalid DNA sequence")#if the input is not valid dna sequence , the code won't work!
help(find_ORFs)
assert find_ORFs("ATGTAACCC")== ["ATGTAA"]

def codon_usage(dna, aa):
    """Return a dictionary containing the percentage of use
    of each DNA codon encoding a specific amino acid,
    rounded to 2 decimal places.
    """
    dna = dna.upper()
    aa = aa.upper()
    counts = {}
    total = 0 # this is useful to compute the percentage of use after

    for i in range(0, len(dna)-2, 3): # loop over each codon
        codon = dna[i:i+3]

        if translate_codon(codon) == aa: # if the result of the function translate_codon == AA put as input
            total += 1 # the usage of this AA gets higher with 1

            if codon in counts: # if the dictionnnary has the AA then add 1 to it's value
                counts[codon] += 1
            else: # if not , just put 1 as value
                counts[codon] = 1
# so now we've created a dictionnary that has different codons for the same AA and their frequencies
# now we will wreate another dictionnary that has the percentage of use not frequency
    usage = {}
    for codon in counts: # loop over each codon in the count dictionnary
        usage[codon] = round(counts[codon] / total * 100, 2) # the value of codon in usage dictionary is the value of the same codon in count dictionary / total
    return usage
help(codon_usage)
assert codon_usage("CTGCTGTTAGCC", "L") == {"CTG": 66.67, "TTA": 33.33}

def read_fasta(path):
    """Read a FASTA file and return a dictionary
    containing identifiers as keys and sequences as values."""

    sequences = {}  # Create an empty dictionary to store the sequences
    identifier = ""  # Create a variable to store the current sequence ID

    with open(path, "r") as f:  # Open the FASTA file in reading mode

        for line in f:  # Read the file line by line

            line = line.strip()  # Remove spaces and newline characters (\n). this is a way of making clean data

            if line.startswith(">"):  # Check if the line is a FASTA header starting with > then

                identifier = line[1:].split()[0]  # Remove ">" and extract the first word (ID)

                sequences[identifier] = ""  # Create a new key (ID) with an empty sequence

            else:  # If the line is not a header, the it shoulf be a dna sequence so it should be a value

                sequences[identifier] += line  # Add the nucleotides to the sequence of the current ID

    return sequences

def write_fasta(records, file):
    """Write a FASTA file from a dictionary of records. each sequence is transfered to a single fasta file """
    with open (file, "w") as f:
      for identifier in records:
         f.write(">" + identifier + "\n")  # Write the FASTA header (>seq1) and move to a new line
         f.write(records[identifier] + "\n") # write the the value of the identifier wich is the sequence in a new line

def write_summary(records, path):

    # Create the CSV file
    with open(path, "w") as f:

        # Write the header
        f.write("id,length,GC%,number_of_ORFs,longest_protein\n")

        # Loop over each identifier in the dictionary
        for identifier in records:

            # Retrieve the corresponding DNA sequence
            seq = records[identifier]

            # Calculate the sequence length
            length = len(seq)

            # Check whether the sequence is valid
            if  validate_dna(seq):

               # Calculate GC content
                gc = gc_content(seq)

                # Find all ORFs
                ORFs = find_ORFs(seq)

                # Count the ORFs
                number_of_ORFs = len(ORFs)

                # Initialize the longest protein length
                longest_protein = 0

                # Loop over each ORF
                for orf in ORFs:

                    # Translate the ORF without its stop codon
                    protein = translate(orf)

                    # Compare protein lengths
                    if len(protein) > longest_protein:
                        longest_protein = len(protein)

            else:
                gc = "non-valid"
                number_of_ORFs = "non-valid"
                longest_protein = "non-valid"

            # Write one CSV line per sequence
            f.write(f"{identifier},{length},{gc},{number_of_ORFs},{longest_protein}\n")










Overwriting seqtools.py


In [ ]:
%%writefile main.py
import seqtools                  # Import your own module
from datetime import datetime    # Import the datetime class

# Ask the user for a DNA sequence
seq = input("Enter a DNA sequence: ")


# Check that the sequence is not empty and contains valid DNA
if seqtools.validate_dna(seq):

    # Get the date and time of the analysis
    now = datetime.now()

    # Print the report
    print("YOUR DNA SEQUENCE SMALL REPORT")
    print("Date and time:", now.strftime("%Y-%m-%d %H:%M:%S"))
    print("DNA sequence:", seq)
    print("Length:", len(seq), "nucleotides")
    print("Nucleotide counts:", seqtools.frequency(seq))
    print("RNA sequence:", seqtools.transcription(seq))
    print("Reverse complement:", seqtools.reverse_complement(seq))
    print("GC content:", seqtools.gc_content(seq), "%")

else:
    print("Invalid DNA sequence. Please enter only A, T, G and C.")


Overwriting main.py


In [ ]:
%run main.py

Enter a DNA sequence: aTGC
YOUR DNA SEQUENCE SMALL REPORT
Date and time: 2026-11-26 15:09:08
DNA sequence: aTGC
Length: 4 nucleotides
Nucleotide counts: {'A': 1, 'C': 1, 'T': 1, 'G': 1}
RNA sequence: AUGC
Reverse complement: GCAT
GC content: 50.0 %


In [ ]:
 %%writefile mainfasta.py

import sys
import os
import seqtools
from datetime import datetime

path = sys.argv[1]
records = seqtools.read_fasta(path)

for identifier in records:
    seq = records[identifier]
    seq = seq.upper()

    print("\nSequence ID:", identifier)

    if seqtools.validate_dna(seq):
        print("DNA sequence is valid!")
        current_date = datetime.now()
        print("\n..........DATE........")
        print(current_date)
        print("\n..........DNA REPORT..........")
        print("Length:", len(seq))
        print("GC content:", seqtools.gc_content(seq), "%")
        print("Nucleotide frequency:", seqtools.frequency(seq))
        print("RNA sequence:", seqtools.transcription(seq))
        print("Reverse complement:", seqtools.reverse_complement(seq))

    else:
        print("Invalid DNA sequence!")

os.makedirs("results", exist_ok=True)
seqtools.write_summary(records, "results/summary.csv")

Overwriting mainfasta.py


In [ ]:
! python mainfasta.py data/sequences.fasta

Help on function translate in module seqtools:

ttrraannssllaattee(seq, start=0)
    this function turns a dna sequence into an amino acid sequence.
    0 is the default value, but you can change it whenever you call the function.

Help on function validate_protein in module seqtools:

vvaalliiddaattee__pprrootteeiinn(protein)
    this function tells you wether the input protein sequence is a protein sequence or not

Help on function has_stop_codon in module seqtools:

hhaass__ssttoopp__ccooddoonn(seq, start=0)
    this function checks wether the input DNA sequence has a STOP codon or not. and you can choose the position of start

Help on function find_ORFs in module seqtools:

ffiinndd__OORRFFss(seq)
    an ORF is open reading frame wich is a DNA sequence that starts with a start condon and ends with STOP codon
    this function finds all the ORFs in a DNA sequence and returns the potential ORF sequences

Help on function codon_usage in 

---
## Milestone 7 — Working with FASTA files (session 7)

In the group notebook, section *Milestone 7*:
1. **Tasks 15–16:** `read_fasta(path)` → dictionary `{id: sequence}` and `write_fasta(records, path)`.
2. **Task 17:** `write_summary(...)`: one CSV line per sequence (id, length, GC%, valid, number of ORFs,
   length of the longest protein).
3. Update `main.py` so it takes the FASTA file from the command line:
   `!python main.py data/sequences.fasta` (`sys.argv`), and writes its outputs in `results/`.

In [ ]:
%run main.py

In [ ]:
!python main.py data/sequences.fasta


Help on function validate_dna in module seqtools:

vvaalliiddaattee__ddnnaa(seq)
    this function checks wether the input sequence is a DNA  sequence or not.
    The function returns True or False

Help on function frequency in module seqtools:

ffrreeqquueennccyy(seq)
    Computes the frequency of the different symbols in a sequence.
    The result is a dictionary whose keys are the symbols
    and whose values are their frequencies.

Help on function aa_frequency in module seqtools:

aaaa__ffrreeqquueennccyy(protein)
    computes the frequency of the amino acids in a protein
    sequence. The result is a dictionary whose keys are the amino acids and whose values are their frequencies.

Help on function gc_content in module seqtools:

ggcc__ccoonntteenntt(seq)
    this function computes the GC content (%) of a DNA sequence. the result is in percent and rounded to 2 decimals

Help on function transcription in module seqtools:

ttrraannss

In [ ]:
!ls



data  drive  main.py  __pycache__  results  sample_data  seqtools_clean.py


In [ ]:
!cat data/sequences.fasta


>gene_alpha synthetic gene, one ORF on the forward strand
AGGTAAGGTGGTTGAGATCTGGACTTTTGACGCCTGGAGCATGCCGCAGTGCTCCTCGAA
AAGTAGCCATGCCTTGGGCTGCTCCCAGGCGCTCCGTTGTTCTAGTGCTTTCACCAGCCA
AGTATTGGACGTTCGAAGTGTTTCCAATACCCCCATGCCGGACATAGTCTGTATCCCACC
CCTCCCCATTTTATTGCTTGTCAGGGAGGGACCGACTAGTGAGGCTCCCACTTCAAAAGG
TCTGGGTCACTTGTGTCTCGCAGCTTATCTGCGGAGAACGGAGTAGCCTGTAAGGGTTAA
GCTAGAAATTTCGGATGAAAGAGTGCTGCTCCCTTG
>gene_beta synthetic gene, ORF on the reverse strand
CACCACTGCAACGCGATGTGCACGTTGTAATTAACACAACCCTGCGCAACAACCCCAACG
CCGGAGGTCGGGGGGCGAGCCTATTCCGTCCGAATACTCGCCCCGACCCTGAAACCTCGT
GTCGCCACCGCTGGCGCCGCTGGCGAGTGGCCGGCAGCAAGTGGCAGCGTCGCAATGAAG
GCGGTGGTATGTGCCGTACCCTTTGTCCACCGACACAGTGGATCCGCGCATAACTAATTG
AGTCGGGAGGAATTGTTCCTATGAAG
>gene_gamma synthetic gene, lower case, two ORFs
tgcgttcgtagctagcactgtcagtatgagaccgtgccacgtgctaatcggaacttcgat
gcgccccctaaaccgaggttacgttttcaagcatttggacacattggtcatagcaatcaa
aaaccgcaggtctcttgctaggtggctttctaataccctagggtaatgattgtgttttgt
aagcagatgatagggggcacgcaacagaaggccgaacgattttc

---
## Milestone 8 — Robustness, final polish and oral presentation (session 8)

1. **Task 18:** define `InvalidSequenceError` and raise it from your functions. In `main.py`, catch errors
   (missing file, invalid records) so that **one bad sequence never stops the whole run**. Write the
   problems to `results/run.log` with a timestamp.
2. Run the complete pipeline on `data/sequences.fasta` (it contains some traps!).
3. Prepare your **oral presentation** (10 min + 5 min questions): live demo in Colab, design choices,
   one difficulty you solved, who did what.

In [ ]:
# Final run: this is what you will show during the oral presentation
!python main.py data/sequences.fasta
!cat results/run.log
!head results/summary.csv

---
## 📝 Contribution log
| Session | Member | What I did |M1|
|---|---|---|---|
| 1 |Ameni |  |
| 2 | Rim| |
|3 | Dihya |